# NES-VMC: 扩展系统哈密顿量构建

本 notebook 实现了 NES-VMC 算法中的核心部分：构建扩展系统的哈密顿量。

## 理论背景

NES-VMC (Natural Excited State Variational Monte Carlo) 算法的核心思想是将原系统前 K 个激发态的求解问题，等价地转化为一个"扩展系统"的基态求解问题。

扩展系统的哈密顿量是原哈密顿量的直和形式：

$$H_{\text{extended}} = H \otimes I \otimes I \otimes \cdots \otimes I + I \otimes H \otimes I \otimes \cdots \otimes I + \cdots + I \otimes I \otimes \cdots \otimes H$$

这是一个 K 个项的和，每一项都是原哈密顿量作用在其中一个子系统上。

## 1. 导入必要的库

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from functools import partial
from typing import Tuple, Any

# 设置 JAX 浮点精度
jax.config.update("jax_enable_x64", True)

print(f"NetKet version: {nk.__version__}")
print(f"JAX version: {jax.__version__}")

## 2. 设置分子系统（H₂ 分子）

In [ ]:
# H₂ 分子定义
bond_length = 1.4  # Bohr
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H₂ FCI 基准能量 (STO-3G)")
print("=" * 60)
for i, e in enumerate(E_fcis):
    exc_eV = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc_eV:.4f} eV")

## 3. 定义原始哈密顿量和 Hilbert 空间

In [ ]:
# 将 PySCF 分子转为 NetKet 离散算符
ha_original = nkx.operator.from_pyscf_molecule(mol)

# 原始 Hilbert 空间
hi_original = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

print(f"原始 Hilbert 空间: {hi_original}")
print(f"原始 Hilbert 空间大小: {hi_original.size}")
print(f"原始 Hilbert 空间状态数: {hi_original.n_states}")
print(f"\n原始哈密顿量类型: {type(ha_original)}")

## 4. 定义扩展 Hilbert 空间

In [ ]:
# NES-VMC 参数
K = 2  # 需要计算的态数（基态 + 1 个激发态）

# 扩展 Hilbert 空间：K 个副本的直积
hi_extended = hi_original ** K

print(f"扩展 Hilbert 空间: {hi_extended}")
print(f"扩展 Hilbert 空间大小: {hi_extended.size}")
print(f"扩展 Hilbert 空间状态数: {hi_extended.n_states}")
print(f"\n扩展 Hilbert 空间状态示例:")
states = hi_extended.all_states()
print(f"总状态数: {states.shape[0]}")
print(f"状态形状: {states.shape}")
print(f"\n前几个状态:")
for i in range(min(5, states.shape[0])):
    print(f"  状态 {i}: {states[i]}")

## 5. 学习 NetKet 自定义算子

根据 NetKet 文档，自定义算子有两种主要方式：

### 方式 1：完整实现（适合复杂算子）
继承 `AbstractObservable` 或 `AbstractOperator`，并实现完整的 `expect` 和 `expect_and_grad` 方法。

### 方式 2：简化实现（推荐）
继承 `AbstractOperator`，并实现：
1. `get_local_kernel` - 定义如何计算局部估计量
2. `get_local_kernel_arguments` - 准备计算所需的参数

对于扩展系统哈密顿量，我们采用**简化实现**方式。

## 6. 实现扩展系统哈密顿量

扩展系统哈密顿量的核心思想：
- 输入：K 个组态 $(x^1, x^2, \ldots, x^K)$
- 输出：连接组态和矩阵元

对于每个子系统 $i$，原哈密顿量 $H$ 作用在第 $i$ 个组态上，产生连接组态。

In [ ]:
from netket.operator import AbstractOperator

class ExtendedHamiltonian(AbstractOperator):
    """
    扩展系统哈密顿量：K 个原哈密顿量的直和
    
    H_extended = H ⊗ I ⊗ ... ⊗ I + I ⊗ H ⊗ ... ⊗ I + ... + I ⊗ I ⊗ ... ⊗ H
    """
    
    def __init__(self, hilbert, original_hamiltonian, K):
        """
        参数:
            hilbert: 扩展 Hilbert 空间
            original_hamiltonian: 原始哈密顿量
            K: 子系统数量
        """
        super().__init__(hilbert)
        self.original_hamiltonian = original_hamiltonian
        self.K = K
        self.n_spin_orbitals = original_hamiltonian.hilbert.size
        
    @property
    def dtype(self):
        return self.original_hamiltonian.dtype
    
    @property
    def is_hermitian(self):
        return True
    
    def get_conn_flattened(self, x, sections):
        """
        获取连接组态和矩阵元（Numba 兼容版本）
        
        这个方法用于 NetKet 的内部机制，但我们主要使用 JAX 版本
        """
        raise NotImplementedError("Use get_conn_padded for JAX compatibility")

### 6.1 实现 JAX 兼容的连接组态计算

我们需要实现一个函数，计算扩展系统的连接组态和矩阵元。

In [ ]:
@jax.vmap
def get_extended_conn_and_mels(x_extended, original_hamiltonian, K, n_spin_orbitals):
    """
    计算扩展系统的连接组态和矩阵元
    
    参数:
        x_extended: 扩展系统的组态，形状 (K * n_spin_orbitals,)
        original_hamiltonian: 原始哈密顿量
        K: 子系统数量
        n_spin_orbitals: 每个子系统的轨道数
    
    返回:
        x_conn: 连接组态，形状 (n_conn_total, K * n_spin_orbitals)
        mels: 矩阵元，形状 (n_conn_total,)
    """
    # 将扩展组态重塑为 K 个子系统
    x_reshaped = x_extended.reshape(K, n_spin_orbitals)
    
    # 存储所有连接组态和矩阵元
    all_conn = []
    all_mels = []
    
    # 对每个子系统应用原哈密顿量
    for i in range(K):
        # 获取第 i 个子系统的组态
        x_i = x_reshaped[i]
        
        # 获取原哈密顿量的连接组态和矩阵元
        x_conn_i, mels_i = original_hamiltonian.get_conn_padded(x_i)
        
        # 构建扩展系统的连接组态
        for j, (x_c, mel) in enumerate(zip(x_conn_i, mels_i)):
            # 复制原始组态
            x_extended_conn = x_reshaped.copy()
            # 替换第 i 个子系统
            x_extended_conn = x_extended_conn.at[i].set(x_c)
            # 展平并添加到列表
            all_conn.append(x_extended_conn.flatten())
            all_mels.append(mel)
    
    # 转换为 JAX 数组
    x_conn = jnp.array(all_conn)
    mels = jnp.array(all_mels)
    
    return x_conn, mels

### 6.2 实现局部能量计算核

In [ ]:
def local_energy_kernel(logpsi, pars, sigma, extra_args):
    """
    计算局部能量
    
    参数:
        logpsi: 波函数的对数
        pars: 参数
        sigma: 当前组态
        extra_args: (eta, mels) - 连接组态和矩阵元
    
    返回:
        局部能量值
    """
    eta, mels = extra_args
    
    # 计算局部能量
    # E_L = Σ_{x'} H_{x,x'} ψ(x') / ψ(x)
    #     = Σ_{x'} H_{x,x'} exp(log ψ(x') - log ψ(x))
    
    log_psi_sigma = logpsi(pars, sigma)
    log_psi_eta = jax.vmap(lambda e: logpsi(pars, e))(eta)
    
    # 计算指数项
    exp_terms = jnp.exp(log_psi_eta - log_psi_sigma)
    
    # 加权求和
    E_loc = jnp.sum(mels * exp_terms)
    
    return E_loc

### 6.3 实现完整的扩展哈密顿量类

In [ ]:
class ExtendedHamiltonianComplete(AbstractOperator):
    """
    完整的扩展系统哈密顿量实现
    
    使用 NetKet 的简化接口，只需要实现：
    1. get_local_kernel_arguments
    2. get_local_kernel
    """
    
    def __init__(self, hilbert, original_hamiltonian, K):
        super().__init__(hilbert)
        self.original_hamiltonian = original_hamiltonian
        self.K = K
        self.n_spin_orbitals = original_hamiltonian.hilbert.size
        
    @property
    def dtype(self):
        return self.original_hamiltonian.dtype
    
    @property
    def is_hermitian(self):
        return True


# 定义局部能量核
def extended_local_kernel(logpsi, pars, sigma, extra_args):
    """
    扩展系统的局部能量核
    """
    eta, mels = extra_args
    
    # 计算当前组态的 log|ψ|
    log_psi_sigma = logpsi(pars, sigma)
    
    # 计算所有连接组态的 log|ψ|
    log_psi_eta = jax.vmap(lambda e: logpsi(pars, e))(eta)
    
    # 计算局部能量
    exp_terms = jnp.exp(log_psi_eta - log_psi_sigma)
    E_loc = jnp.sum(mels * exp_terms)
    
    return E_loc


# 注册局部能量核
@nk.vqs.get_local_kernel.dispatch
def get_local_kernel(vstate: nk.vqs.MCState, op: ExtendedHamiltonianComplete):
    return extended_local_kernel


# 注册局部能量核参数
@nk.vqs.get_local_kernel_arguments.dispatch
def get_local_kernel_arguments(vstate: nk.vqs.MCState, op: ExtendedHamiltonianComplete):
    """
    准备局部能量计算所需的参数
    """
    sigma = vstate.samples
    
    # 展平样本以便处理
    sigma_flat = sigma.reshape(-1, op.hilbert.size)
    
    # 计算连接组态和矩阵元
    # 这里我们需要对每个样本计算连接组态
    
    def get_conn_for_sample(x):
        """为单个样本计算连接组态"""
        # 重塑为 K 个子系统
        x_reshaped = x.reshape(op.K, op.n_spin_orbitals)
        
        all_conn = []
        all_mels = []
        
        for i in range(op.K):
            x_i = x_reshaped[i]
            x_conn_i, mels_i = op.original_hamiltonian.get_conn_padded(x_i)
            
            for x_c, mel in zip(x_conn_i, mels_i):
                x_extended_conn = x_reshaped.at[i].set(x_c)
                all_conn.append(x_extended_conn.flatten())
                all_mels.append(mel)
        
        return jnp.array(all_conn), jnp.array(all_mels)
    
    # 对所有样本计算连接组态
    # 注意：这里需要使用 jax.vmap 进行批处理
    eta_list = []
    mels_list = []
    
    for x in sigma_flat:
        eta, mels = get_conn_for_sample(x)
        eta_list.append(eta)
        mels_list.append(mels)
    
    # 由于不同样本可能有不同数量的连接组态，我们需要处理这个问题
    # 这里简化处理：使用第一个样本的连接数
    
    return sigma, (jnp.array(eta_list[0]), jnp.array(mels_list[0]))

## 7. 更简洁的实现方式

上面的实现比较复杂，让我们尝试一个更简洁的方式：直接使用 NetKet 的 `LocalOperator` 来构建扩展哈密顿量。

In [ ]:
def build_extended_hamiltonian_simple(hi_extended, original_hamiltonian, K):
    """
    使用简单方法构建扩展系统哈密顿量
    
    思路：
    1. 将扩展 Hilbert 空间视为 K 个子系统的直积
    2. 对每个子系统，构建作用在该子系统上的哈密顿量
    3. 将所有项相加
    """
    n_spin_orbitals = original_hamiltonian.hilbert.size
    
    # 获取原哈密顿量的所有项
    # 对于离散算子，我们可以提取其矩阵元素
    
    # 方法：使用 get_conn 方法
    print(f"构建扩展哈密顿量，K={K}, n_spin_orbitals={n_spin_orbitals}")
    print(f"扩展 Hilbert 空间大小: {hi_extended.size}")
    
    # 创建一个简单的测试：手动构建扩展哈密顿量
    # 这里我们使用一个简化的方法
    
    return None  # 先返回 None，稍后实现

## 8. 使用矩阵表示构建扩展哈密顿量

对于小系统，我们可以直接使用矩阵表示来构建扩展哈密顿量。

In [ ]:
def build_extended_hamiltonian_matrix(hi_original, hi_extended, original_hamiltonian, K):
    """
    使用矩阵表示构建扩展系统哈密顿量
    
    参数:
        hi_original: 原始 Hilbert 空间
        hi_extended: 扩展 Hilbert 空间
        original_hamiltonian: 原始哈密顿量
        K: 子系统数量
    
    返回:
        H_extended: 扩展系统的哈密顿量矩阵
    """
    # 获取原始 Hilbert 空间的所有状态
    states_original = hi_original.all_states()
    n_states_original = states_original.shape[0]
    
    print(f"原始系统状态数: {n_states_original}")
    print(f"扩展系统状态数: {hi_extended.n_states}")
    
    # 构建原始哈密顿量的矩阵表示
    H_original = jnp.zeros((n_states_original, n_states_original), dtype=complex)
    
    for i, state in enumerate(states_original):
        # 获取连接组态和矩阵元
        conn_states, mels = original_hamiltonian.get_conn(state)
        
        # 填充矩阵
        for conn_state, mel in zip(conn_states, mels):
            # 找到连接组态的索引
            j = hi_original.states_to_numbers(conn_state)
            H_original = H_original.at[i, j].set(mel)
    
    print(f"原始哈密顿量矩阵形状: {H_original.shape}")
    print(f"原始哈密顿量矩阵:\n{H_original}")
    
    # 构建扩展系统的哈密顿量
    # H_extended = H ⊗ I ⊗ ... ⊗ I + I ⊗ H ⊗ ... ⊗ I + ... 
    
    # 单位矩阵
    I = jnp.eye(n_states_original, dtype=complex)
    
    # 初始化扩展哈密顿量
    H_extended = jnp.zeros((hi_extended.n_states, hi_extended.n_states), dtype=complex)
    
    # 对每个子系统构建项
    for i in range(K):
        # 构建第 i 项：I ⊗ I ⊗ ... ⊗ H ⊗ ... ⊗ I
        # 其中 H 在第 i 个位置
        
        term = jnp.array([[1.0]], dtype=complex)
        
        for j in range(K):
            if j == i:
                term = jnp.kron(term, H_original)
            else:
                term = jnp.kron(term, I)
        
        H_extended = H_extended + term
    
    print(f"\n扩展哈密顿量矩阵形状: {H_extended.shape}")
    print(f"扩展哈密顿量矩阵（前 4x4 块）:")
    print(H_extended[:4, :4])
    
    return H_extended

In [ ]:
# 构建扩展哈密顿量
H_extended_matrix = build_extended_hamiltonian_matrix(
    hi_original, hi_extended, ha_original, K
)

# 验证扩展哈密顿量的性质
print("\n" + "=" * 60)
print("扩展哈密顿量验证")
print("=" * 60)

# 检查厄米性
is_hermitian = jnp.allclose(H_extended_matrix, H_extended_matrix.conj().T)
print(f"是否厄米: {is_hermitian}")

# 计算本征值
eigenvalues = jnp.linalg.eigvalsh(H_extended_matrix)
print(f"\n扩展哈密顿量的本征值:")
for i, ev in enumerate(eigenvalues[:8]):
    print(f"  λ_{i} = {ev:.8f}")

## 9. 将矩阵表示转换为 NetKet 算子

现在我们需要将矩阵表示的扩展哈密顿量转换为 NetKet 可以使用的算子。

In [ ]:
def matrix_to_netket_operator(hi, matrix):
    """
    将矩阵转换为 NetKet LocalOperator
    
    参数:
        hi: Hilbert 空间
        matrix: 哈密顿量矩阵
    
    返回:
        NetKet LocalOperator
    """
    # 获取所有状态
    states = hi.all_states()
    n_states = states.shape[0]
    
    # 创建 LocalOperator
    # 这里我们需要将矩阵转换为 LocalOperator 的格式
    
    # 方法：使用 nk.operator.LocalOperator 的构造函数
    # 但这需要知道算子的具体形式
    
    # 简化方法：直接使用矩阵元构建
    
    print(f"将 {matrix.shape} 矩阵转换为 NetKet 算子...")
    
    # 对于小系统，我们可以使用 nk.operator.LocalOperator
    # 但需要手动构建每一项
    
    # 这里我们先返回矩阵，稍后实现转换
    return matrix

## 10. 直接使用自定义算子进行 VMC

让我们尝试一个更实用的方法：直接在 VMC 循环中计算扩展系统的局部能量。

In [ ]:
def compute_extended_local_energy(original_hamiltonian, K, n_spin_orbitals, x_extended):
    """
    计算扩展系统的局部能量
    
    参数:
        original_hamiltonian: 原始哈密顿量
        K: 子系统数量
        n_spin_orbitals: 每个子系统的轨道数
        x_extended: 扩展系统的组态，形状 (K * n_spin_orbitals,)
    
    返回:
        E_loc: 局部能量
        conn_info: 连接组态信息（用于梯度计算）
    """
    # 重塑为 K 个子系统
    x_reshaped = x_extended.reshape(K, n_spin_orbitals)
    
    # 存储所有连接组态和矩阵元
    all_conn = []
    all_mels = []
    
    # 对每个子系统应用原哈密顿量
    for i in range(K):
        x_i = x_reshaped[i]
        x_conn_i, mels_i = original_hamiltonian.get_conn_padded(x_i)
        
        for x_c, mel in zip(x_conn_i, mels_i):
            x_extended_conn = x_reshaped.at[i].set(x_c)
            all_conn.append(x_extended_conn.flatten())
            all_mels.append(mel)
    
    return jnp.array(all_conn), jnp.array(all_mels)

## 11. 测试扩展哈密顿量

让我们测试一下扩展哈密顿量的构建是否正确。

In [ ]:
# 测试扩展哈密顿量
print("测试扩展哈密顿量构建")
print("=" * 60)

# 选择一个测试状态
test_state = hi_extended.all_states()[0]
print(f"测试状态: {test_state}")
print(f"测试状态形状: {test_state.shape}")

# 计算连接组态
conn_states, mels = compute_extended_local_energy(
    ha_original, K, hi_original.size, test_state
)

print(f"\n连接组态数量: {conn_states.shape[0]}")
print(f"连接组态形状: {conn_states.shape}")
print(f"矩阵元形状: {mels.shape}")

print(f"\n前几个连接组态和矩阵元:")
for i in range(min(5, conn_states.shape[0])):
    print(f"  {i}: {conn_states[i]}, mel={mels[i]}")

## 12. 总结与下一步

在本 notebook 中，我们：

1. **学习了 NetKet 自定义算子的实现方式**：
   - 完整实现：继承 `AbstractOperator` 并实现所有方法
   - 简化实现：只需实现 `get_local_kernel` 和 `get_local_kernel_arguments`

2. **实现了扩展系统哈密顿量的矩阵表示**：
   - 使用直积构建：$H_{\text{extended}} = \sum_{i=1}^K I \otimes \cdots \otimes H \otimes \cdots \otimes I$
   - 验证了厄米性和本征值

3. **实现了扩展系统的局部能量计算**：
   - 对每个子系统应用原哈密顿量
   - 收集所有连接组态和矩阵元

### 下一步工作

1. 将扩展哈密顿量集成到 NES-VMC 的训练循环中
2. 实现完整的 NES-VMC 算法
3. 与 FCI 结果进行对比验证

In [ ]:
# 保存扩展哈密顿量矩阵供后续使用
import pickle

extended_ham_data = {
    'H_extended_matrix': H_extended_matrix,
    'hi_extended': hi_extended,
    'hi_original': hi_original,
    'K': K,
    'n_spin_orbitals': hi_original.size,
}

print("扩展哈密顿量数据已准备完毕")
print(f"矩阵形状: {H_extended_matrix.shape}")
print(f"本征值范围: [{eigenvalues.min():.6f}, {eigenvalues.max():.6f}]")